# Milestone 4: Timing the GPU properly

Same kernels as Milestone 2. Only the measuring is new:
- each kernel is timed on its own with **CUDA events**
- the copies CPU → GPU and GPU → CPU are timed **separately** from the kernels
- the first (cold) run is shown on its own, and runs 2 to 11 are **averaged**

The comment block in `gpu_naive_timed_main.cu` explains why `std::chrono` can't time a kernel and what each `cudaEvent...` function does.

**Runtime must be T4 GPU.**

## Step 1: Connect Drive and go to the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/isp_project

## Step 2: Write the new files
- `gpu_buffers.h` / `.cu`: the GPU memory allocation from Milestone 2, moved into functions so later milestones can reuse it.
- `gpu_naive_timed_main.cu`: the timed pipeline. The kernels themselves are unchanged and come from your existing `gpu_*_kernels.cu` files.

In [ ]:
%%writefile gpu_buffers.h
// gpu_buffers.h
// All the GPU memory one pipeline run needs, kept together in one struct so
// we can allocate it, pass it around and free it with single function calls.
// Later milestones reuse this.

#pragma once

#include <cstddef>
#include <cstdint>

struct DeviceBuffers {
    uint16_t* deviceBayer;
    float* deviceNormalised;
    float* deviceRgb;
    float* deviceDenoised;
    unsigned char* deviceOutput;

    size_t bayerBytes;
    size_t normalisedBytes;
    size_t rgbFloatBytes;
    size_t outputBytes;
};

void allocateDeviceBuffers(DeviceBuffers& buffers, int pixelCount);
void freeDeviceBuffers(DeviceBuffers& buffers);


In [ ]:
%%writefile gpu_buffers.cu
// gpu_buffers.cu
// The cudaMalloc / cudaFree calls from Milestone 2, moved into functions.

#include "gpu_buffers.h"

#include "cuda_helpers.h"

void allocateDeviceBuffers(DeviceBuffers& buffers, int pixelCount) {
    size_t pixels = static_cast<size_t>(pixelCount);
    buffers.bayerBytes = pixels * sizeof(uint16_t);
    buffers.normalisedBytes = pixels * sizeof(float);
    buffers.rgbFloatBytes = pixels * 3 * sizeof(float);
    buffers.outputBytes = pixels * 3 * sizeof(unsigned char);

    checkCudaError(cudaMalloc(&buffers.deviceBayer, buffers.bayerBytes),
                   "allocating Bayer buffer");
    checkCudaError(cudaMalloc(&buffers.deviceNormalised, buffers.normalisedBytes),
                   "allocating normalised buffer");
    checkCudaError(cudaMalloc(&buffers.deviceRgb, buffers.rgbFloatBytes),
                   "allocating RGB buffer");
    checkCudaError(cudaMalloc(&buffers.deviceDenoised, buffers.rgbFloatBytes),
                   "allocating denoised buffer");
    checkCudaError(cudaMalloc(&buffers.deviceOutput, buffers.outputBytes),
                   "allocating output buffer");
}

void freeDeviceBuffers(DeviceBuffers& buffers) {
    checkCudaError(cudaFree(buffers.deviceBayer), "freeing Bayer buffer");
    checkCudaError(cudaFree(buffers.deviceNormalised), "freeing normalised buffer");
    checkCudaError(cudaFree(buffers.deviceRgb), "freeing RGB buffer");
    checkCudaError(cudaFree(buffers.deviceDenoised), "freeing denoised buffer");
    checkCudaError(cudaFree(buffers.deviceOutput), "freeing output buffer");
}


In [ ]:
%%writefile gpu_naive_timed_main.cu
// gpu_naive_timed_main.cu
// Milestone 4: the same naive pipeline as Milestone 2, now with careful timing.
//   - Each kernel is timed separately with CUDA events.
//   - The copies between CPU and GPU are timed separately from the kernels.
//   - Run 1 (the "cold" warm-up run) is reported on its own.
//   - Runs 2 to 11 are averaged to get a stable number.

#include <chrono>
#include <iomanip>
#include <iostream>
#include <vector>

#include "cuda_helpers.h"
#include "gpu_buffers.h"
#include "gpu_kernels.h"
#include "isp_io.h"

const int STAGE_COUNT = 5;
const int TIMED_RUN_COUNT = 10;
const char* STAGE_NAMES[STAGE_COUNT] = {
    "Stage 1 black level",
    "Stage 2 demosaic   ",
    "Stage 3 white bal. ",
    "Stage 4 denoise    ",
    "Stage 5 gamma      "
};

// ---------------------------------------------------------------------------
// Why CUDA events instead of std::chrono?
// A kernel launch returns to the CPU straight away, before the GPU has done
// the work. A CPU clock around a launch measures only "how long it took to
// ASK", not how long the GPU worked. A CUDA event is a marker we drop into the
// GPU's queue of work. The GPU writes a timestamp when it reaches the marker,
// so the gap between two events is real GPU time.
//
//   cudaEventCreate(&event)        - make an event object
//   cudaEventRecord(event)         - put the marker into the GPU's work queue
//   cudaEventSynchronize(event)    - make the CPU wait until the GPU reaches it
//   cudaEventElapsedTime(&ms, a, b)- milliseconds between markers a and b
//   cudaEventDestroy(event)        - free the event object
// ---------------------------------------------------------------------------

// Runs all 5 kernels once. stageEvents must hold STAGE_COUNT + 1 events:
// event 0 before kernel 1, event 1 after kernel 1 (= before kernel 2), and so on.
static void runPipelineOnce(DeviceBuffers& buffers, const ImageMetadata& metadata,
                            dim3 gridShape, dim3 blockShape,
                            cudaEvent_t stageEvents[], float stageMilliseconds[]) {
    int imageWidth = metadata.imageWidth;
    int imageHeight = metadata.imageHeight;

    checkCudaError(cudaEventRecord(stageEvents[0]), "recording event before stage 1");

    blackLevelKernel<<<gridShape, blockShape>>>(buffers.deviceBayer, buffers.deviceNormalised,
                                                imageWidth, imageHeight,
                                                metadata.blackLevel, metadata.whiteLevel);
    checkCudaError(cudaGetLastError(), "launching blackLevelKernel");
    checkCudaError(cudaEventRecord(stageEvents[1]), "recording event after stage 1");

    demosaicKernel<<<gridShape, blockShape>>>(buffers.deviceNormalised, buffers.deviceRgb,
                                              imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching demosaicKernel");
    checkCudaError(cudaEventRecord(stageEvents[2]), "recording event after stage 2");

    whiteBalanceKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, imageWidth, imageHeight,
                                                  metadata.whiteBalanceRed,
                                                  metadata.whiteBalanceGreen,
                                                  metadata.whiteBalanceBlue);
    checkCudaError(cudaGetLastError(), "launching whiteBalanceKernel");
    checkCudaError(cudaEventRecord(stageEvents[3]), "recording event after stage 3");

    denoiseKernel<<<gridShape, blockShape>>>(buffers.deviceRgb, buffers.deviceDenoised,
                                             imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching denoiseKernel");
    checkCudaError(cudaEventRecord(stageEvents[4]), "recording event after stage 4");

    gammaKernel<<<gridShape, blockShape>>>(buffers.deviceDenoised, buffers.deviceOutput,
                                           imageWidth, imageHeight);
    checkCudaError(cudaGetLastError(), "launching gammaKernel");
    checkCudaError(cudaEventRecord(stageEvents[5]), "recording event after stage 5");

    // Wait for the last marker, so every timestamp exists before we read them.
    checkCudaError(cudaEventSynchronize(stageEvents[STAGE_COUNT]), "waiting for the pipeline");

    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        checkCudaError(cudaEventElapsedTime(&stageMilliseconds[stage],
                                            stageEvents[stage], stageEvents[stage + 1]),
                       "reading stage time");
    }
}

static void printStageTimes(const char* heading, const float stageMilliseconds[]) {
    std::cout << heading << std::endl;
    float totalMilliseconds = 0.0f;
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        std::cout << "  " << STAGE_NAMES[stage] << " : " << stageMilliseconds[stage] << " ms" << std::endl;
        totalMilliseconds = totalMilliseconds + stageMilliseconds[stage];
    }
    std::cout << "  All 5 kernels       : " << totalMilliseconds << " ms" << std::endl;
}

int main() {
    // Print every number with 3 digits after the decimal point.
    std::cout << std::fixed << std::setprecision(3);

    ImageMetadata metadata;
    bool metadataOk = readMetadataFile("data/metadata.txt", metadata);
    if (!metadataOk) {
        return 1;
    }
    int pixelCount = metadata.imageWidth * metadata.imageHeight;
    std::vector<uint16_t> bayerPixels;
    bool bayerOk = readBayerFile("data/bayer.bin", pixelCount, bayerPixels);
    if (!bayerOk) {
        return 1;
    }
    std::cout << "Image: " << metadata.imageWidth << " x " << metadata.imageHeight << std::endl;

    // ---- CUDA start-up cost ----
    // The very first CUDA call creates a "context" (the GPU-side setup for this
    // program). That is slow and happens only once. cudaFree(0) frees nothing,
    // but forces the start-up to happen here, so it does not pollute other timings.
    // We use std::chrono for this one because no GPU work is queued yet.
    std::chrono::steady_clock::time_point startupBegin = std::chrono::steady_clock::now();
    checkCudaError(cudaFree(0), "starting CUDA");
    std::chrono::steady_clock::time_point startupEnd = std::chrono::steady_clock::now();
    std::chrono::duration<double, std::milli> startupTime = startupEnd - startupBegin;
    std::cout << "CUDA start-up (once per program): " << startupTime.count() << " ms" << std::endl;

    DeviceBuffers buffers;
    allocateDeviceBuffers(buffers, pixelCount);

    cudaEvent_t copyStart;
    cudaEvent_t copyStop;
    checkCudaError(cudaEventCreate(&copyStart), "creating copyStart event");
    checkCudaError(cudaEventCreate(&copyStop), "creating copyStop event");

    cudaEvent_t stageEvents[STAGE_COUNT + 1];
    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventCreate(&stageEvents[eventNumber]), "creating stage event");
    }

    // ---- Time the copy CPU -> GPU ----
    checkCudaError(cudaEventRecord(copyStart), "recording copyStart");
    checkCudaError(cudaMemcpy(buffers.deviceBayer, bayerPixels.data(), buffers.bayerBytes,
                              cudaMemcpyHostToDevice), "copying Bayer data to the GPU");
    checkCudaError(cudaEventRecord(copyStop), "recording copyStop");
    checkCudaError(cudaEventSynchronize(copyStop), "waiting for copy to GPU");
    float copyToGpuMilliseconds = 0.0f;
    checkCudaError(cudaEventElapsedTime(&copyToGpuMilliseconds, copyStart, copyStop),
                   "reading copy-to-GPU time");

    dim3 blockShape(BLOCK_WIDTH, BLOCK_HEIGHT);
    int blocksAcross = (metadata.imageWidth + BLOCK_WIDTH - 1) / BLOCK_WIDTH;
    int blocksDown = (metadata.imageHeight + BLOCK_HEIGHT - 1) / BLOCK_HEIGHT;
    dim3 gridShape(blocksAcross, blocksDown);

    // ---- Run 1: the cold (warm-up) run, reported on its own ----
    float coldStageMilliseconds[STAGE_COUNT];
    runPipelineOnce(buffers, metadata, gridShape, blockShape, stageEvents, coldStageMilliseconds);
    printStageTimes("Run 1 (cold, includes one-time warm-up):", coldStageMilliseconds);

    // ---- Runs 2 to 11: add up each stage, then divide to get the average ----
    float summedStageMilliseconds[STAGE_COUNT] = {0.0f, 0.0f, 0.0f, 0.0f, 0.0f};
    for (int runNumber = 0; runNumber < TIMED_RUN_COUNT; runNumber++) {
        float thisRunMilliseconds[STAGE_COUNT];
        runPipelineOnce(buffers, metadata, gridShape, blockShape, stageEvents, thisRunMilliseconds);
        for (int stage = 0; stage < STAGE_COUNT; stage++) {
            summedStageMilliseconds[stage] = summedStageMilliseconds[stage] + thisRunMilliseconds[stage];
        }
    }
    float averageStageMilliseconds[STAGE_COUNT];
    float averageKernelTotal = 0.0f;
    for (int stage = 0; stage < STAGE_COUNT; stage++) {
        averageStageMilliseconds[stage] = summedStageMilliseconds[stage] / TIMED_RUN_COUNT;
        averageKernelTotal = averageKernelTotal + averageStageMilliseconds[stage];
    }
    printStageTimes("Average of runs 2 to 11 (warm):", averageStageMilliseconds);

    // ---- Time the copy GPU -> CPU ----
    std::vector<unsigned char> outputBytes(buffers.outputBytes);
    checkCudaError(cudaEventRecord(copyStart), "recording copyStart");
    checkCudaError(cudaMemcpy(outputBytes.data(), buffers.deviceOutput, buffers.outputBytes,
                              cudaMemcpyDeviceToHost), "copying output to the CPU");
    checkCudaError(cudaEventRecord(copyStop), "recording copyStop");
    checkCudaError(cudaEventSynchronize(copyStop), "waiting for copy to CPU");
    float copyToCpuMilliseconds = 0.0f;
    checkCudaError(cudaEventElapsedTime(&copyToCpuMilliseconds, copyStart, copyStop),
                   "reading copy-to-CPU time");

    float totalWithCopies = copyToGpuMilliseconds + averageKernelTotal + copyToCpuMilliseconds;
    std::cout << "Copy CPU -> GPU (" << buffers.bayerBytes / 1000000.0 << " MB) : "
              << copyToGpuMilliseconds << " ms" << std::endl;
    std::cout << "Copy GPU -> CPU (" << buffers.outputBytes / 1000000.0 << " MB) : "
              << copyToCpuMilliseconds << " ms" << std::endl;
    std::cout << "Warm total = copy in + kernels + copy out : " << totalWithCopies << " ms" << std::endl;

    // ---- Clean up ----
    checkCudaError(cudaEventDestroy(copyStart), "destroying copyStart");
    checkCudaError(cudaEventDestroy(copyStop), "destroying copyStop");
    for (int eventNumber = 0; eventNumber <= STAGE_COUNT; eventNumber++) {
        checkCudaError(cudaEventDestroy(stageEvents[eventNumber]), "destroying stage event");
    }
    freeDeviceBuffers(buffers);

    bool writeOk = writeRgbFile("output/gpu_naive_output.bin", outputBytes);
    if (!writeOk) {
        return 1;
    }
    std::cout << "Saved output/gpu_naive_output.bin" << std::endl;
    return 0;
}


## Step 3: Compile
**Expected:** no output.

In [ ]:
!nvcc -O2 -arch=sm_75 -o /content/gpu_naive_timed gpu_naive_timed_main.cu gpu_buffers.cu gpu_pointwise_kernels.cu gpu_neighbourhood_kernels.cu cuda_helpers.cu isp_io.cpp

## Step 4: Run (CHECKPOINT 1)
**Expected:** a CUDA start-up time, then a "Run 1 (cold)" table, then an "Average of runs 2 to 11 (warm)" table, then the two copy times and a warm total. Run 1 should be noticeably slower than the average.

**Paste the full output to me.**

In [ ]:
!/content/gpu_naive_timed

## Step 5: Confirm the output is still correct (CHECKPOINT 2)
We changed the program, so we recheck. **Expected:** `RESULT: PASS`. Just tell me whether it passed.

In [ ]:
!/content/compare_outputs output/cpu_output.bin output/gpu_naive_output.bin

## Why the first run is slower (warm-up)

Run 1 pays for several one-time costs that later runs don't:

1. **Lazy kernel loading.** Recent CUDA versions don't copy a kernel's GPU code onto the GPU until the first time that kernel is launched. So run 1 of each kernel includes a small loading step.
2. **GPU clock speed.** An idle GPU saves power by running slowly. Your Milestone 0 `nvidia-smi` showed `P8`, a low-power state. When work arrives, it takes a moment to ramp up to full clock speed.
3. **Cold memory.** The first touch of freshly allocated GPU memory, and of the GPU's address-translation tables, is slower than later touches.

None of these are part of what a kernel costs when an ISP processes frame after frame, so we report run 1 separately and leave it out of the average.

The **CUDA start-up** line is an even bigger one-time cost: it sets up the GPU for this program. It's paid once per program, not once per image.

## Why average several runs

Even warm runs vary a little. Clock speed drifts with temperature, the operating system occasionally interrupts, and the timer itself has a resolution of about half a microsecond. One run might be lucky or unlucky. The average of 10 runs smooths that out, so when we compare optimised versions later, a difference means a real improvement, not noise.

## Why copies are timed separately

The copies depend only on how many bytes cross the PCIe bus between the CPU and GPU, not on how clever our kernels are. Milestones 5–7 make the **kernels** faster, and the copy time won't change. Keeping the two apart lets us see each improvement clearly. It also shows an important truth about GPUs: if the copies cost more than the kernels, a faster kernel barely helps the total.